# Desinformação: a resolução do CONSEPE que ninguém assinou

OWASP Top 10 para LLMs 2025 — **LLM09 Misinformation**

Simulador didático (`campus/`), sem rede e sem chaves. Cenário do Campus Aurora (universidade fictícia, dados sintéticos).

In [1]:
import os, sys
_dir = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isdir(os.path.join(_dir, "campus")):
        break
    _dir = os.path.dirname(_dir)
sys.path.insert(0, _dir)

from campus import dados, defesas, ferramentas, narrar  # noqa: E402
from campus.llm import SimLLM  # noqa: E402

print("Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.")

Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.


## Cenário
Um aluno pede para aproveitar uma disciplina cursada em outra instituição. O assistente nega o pedido citando uma resolução que não existe, com número, artigo e ano plausíveis.

In [2]:
import re

# Registro oficial (sintético): o que existe e o assunto de cada norma.
RESOLUCOES_OFICIAIS = {
    "Resolução CONSEPE nº 008/2024": "calendário acadêmico",
    "Resolução CONSEPE nº 012/2025": "concessão de bolsas",
}
PERGUNTA = "Posso aproveitar a disciplina de Estatística que cursei em outra instituição?"

# O que um modelo sem grounding devolveu (texto fluente, citação inventada):
RESPOSTA_DO_MODELO = ("Não é possível, conforme a Resolução CONSEPE nº 118/2022, artigo 7º, que veda "
                      "o aproveitamento entre programas de áreas distintas.")

def citacoes(texto):
    return re.findall(r"Resolução CONSEPE nº \d{3}/\d{4}", texto)

## Sem defesa
A resposta vai direto ao aluno.

In [3]:
print(RESPOSTA_DO_MODELO)
citadas = citacoes(RESPOSTA_DO_MODELO)
print("citações:", citadas)
assert citadas and not any(c in RESOLUCOES_OFICIAIS for c in citadas)  # a norma não existe

Não é possível, conforme a Resolução CONSEPE nº 118/2022, artigo 7º, que veda o aproveitamento entre programas de áreas distintas.
citações: ['Resolução CONSEPE nº 118/2022']


## Com defesa
Antes de entregar, toda citação é conferida contra o registro oficial. Citação inexistente → a resposta não sai; o pedido vai para análise humana.

In [4]:
def entregar(resposta):
    inexistentes = [c for c in citacoes(resposta) if c not in RESOLUCOES_OFICIAIS]
    if inexistentes:
        return ("Não encontrei base oficial para responder a esse pedido; ele foi encaminhado "
                "para análise da secretaria. (citação descartada: {})".format(", ".join(inexistentes)))
    return resposta

resposta_defendida = entregar(RESPOSTA_DO_MODELO)
print(resposta_defendida)
assert "análise da secretaria" in resposta_defendida

Não encontrei base oficial para responder a esse pedido; ele foi encaminhado para análise da secretaria. (citação descartada: Resolução CONSEPE nº 118/2022)


## O que observar
- Número, artigo e ano dão tom de norma oficial: é exatamente isso que faz a resposta parecer verdadeira.
- Como no caso Moffatt v. Air Canada (2024), quem publica o assistente responde pelo que ele diz.
- A defesa é grounding com verificação externa: só afirmar o que está numa fonte oficial e admitir "não sei" quando não está. A verificação fica fora do modelo.